# LeetCode 2284 — Sender With Largest Word Count bằng Python

Link đề: https://leetcode.com/problems/sender-with-largest-word-count/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def largestWordCount(self, messages: List[str], senders: List[str]) -> str:
        ...
```

## 1. Mục tiêu học

1. **Gom nhóm và cộng dồn** theo khóa bằng `dict` / `collections.Counter` / `defaultdict(int)`.
2. Đếm số từ trong một câu "chuẩn" (một dấu cách giữa các từ, không có dấu cách đầu/cuối): `s.count(" ") + 1` hay `len(s.split())`, và hiểu khi nào hai cách này **khác nhau**.
3. Chọn phần tử tốt nhất theo **khóa ghép** `(số từ, tên)`: tiêu chí chính lớn nhất, hòa thì tên lớn nhất theo thứ tự từ điển — gói gọn trong `max(..., key=lambda k: (cnt[k], k))`.
4. Hiểu **thứ tự từ điển trong Python** so sánh theo mã Unicode: `'A' (65) < 'Z' (90) < 'a' (97)` ⇒ chữ hoa đứng **trước** chữ thường, đúng như ghi chú của đề.
5. Nhận ra cái bẫy của `max(cnt, key=cnt.get)`: khi hòa nó trả về khóa **xuất hiện trước**, không phải tên lớn nhất.

## 2. Đề bài & ví dụ

Có nhật ký chat gồm `n` tin nhắn: `messages[i]` do `senders[i]` gửi. Mỗi tin nhắn là danh sách các từ phân tách bởi **đúng một** dấu cách, không có dấu cách ở đầu/cuối. **Số từ** của một người gửi là tổng số từ trong mọi tin nhắn của người đó (một người có thể gửi nhiều tin).

Trả về người gửi có số từ **lớn nhất**. Nếu nhiều người cùng lớn nhất, trả về người có tên **lớn nhất theo thứ tự từ điển**.

Lưu ý: chữ hoa đứng trước chữ thường trong thứ tự từ điển; `"Alice"` và `"alice"` là hai người khác nhau.

**Ví dụ 1:** `messages = ["Hello userTwooo","Hi userThree","Wonderful day Alice","Nice day userThree"]`, `senders = ["Alice","userTwo","userThree","Alice"]` → `"Alice"` (Alice 2 + 3 = 5 từ; userTwo 2; userThree 3).

**Ví dụ 2:** `messages = ["How is leetcode for everyone","Leetcode is useful for practice"]`, `senders = ["Bob","Charlie"]` → `"Charlie"` (hòa 5–5, `"Charlie" > "Bob"`).

**Ràng buộc:** `1 ≤ n ≤ 10⁴`; `1 ≤ len(messages[i]) ≤ 100`; `1 ≤ len(senders[i]) ≤ 10`; tin nhắn gồm chữ cái hoa/thường và dấu cách; tên chỉ gồm chữ cái.

In [1]:
from typing import List
from collections import Counter, defaultdict
import heapq, random, string, time

examples = [
    (["Hello userTwooo", "Hi userThree", "Wonderful day Alice", "Nice day userThree"],
     ["Alice", "userTwo", "userThree", "Alice"], "Alice"),
    (["How is leetcode for everyone", "Leetcode is useful for practice"], ["Bob", "Charlie"], "Charlie"),
]
for msgs, snd, exp in examples:
    for m_, s_ in zip(msgs, snd):
        print(f"   {s_:>10}: {m_!r}")
    print("   -> kỳ vọng:", exp)

        Alice: 'Hello userTwooo'
      userTwo: 'Hi userThree'
    userThree: 'Wonderful day Alice'
        Alice: 'Nice day userThree'
   -> kỳ vọng: Alice
          Bob: 'How is leetcode for everyone'
      Charlie: 'Leetcode is useful for practice'
   -> kỳ vọng: Charlie


## 3. Kiến thức nền

### 3.1 Đếm từ trong một câu chuẩn

Câu có `w` từ và đúng một dấu cách giữa hai từ liền nhau ⇒ có `w − 1` dấu cách ⇒ `w = s.count(" ") + 1`. Cách khác: `len(s.split())` (tách theo khoảng trắng bất kỳ, bỏ chuỗi rỗng) hoặc `len(s.split(" "))` (tách đúng theo một dấu cách).

In [2]:
for s in ["Hello userTwooo", "Hi", "Wonderful day Alice"]:
    print(repr(s), s.count(" ") + 1, len(s.split()), len(s.split(" ")))

'Hello userTwooo' 2 2 2
'Hi' 1 1 1
'Wonderful day Alice' 3 3 3


### 3.2 Khi nào ba cách đếm khác nhau?

Chỉ khi câu **không chuẩn** (dấu cách thừa, ở đầu/cuối). Đề đảm bảo câu chuẩn nên ba cách cho cùng kết quả; nhưng hiểu sự khác biệt giúp tránh bug khi đề thay đổi.

In [3]:
for s in ["a  b", " a b", "a b ", "   "]:
    print(f"{s!r:<8} count+1={s.count(' ') + 1}  split()={len(s.split())}  split(' ')={len(s.split(' '))}")

'a  b'   count+1=3  split()=2  split(' ')=3
' a b'   count+1=3  split()=2  split(' ')=3
'a b '   count+1=3  split()=2  split(' ')=3
'   '    count+1=4  split()=0  split(' ')=4


`split()` không đối số là cách **an toàn nhất** với dữ liệu bẩn; `count(" ") + 1` nhanh nhất (không tạo danh sách từ) và đúng với dữ liệu chuẩn của đề.

### 3.3 Cộng dồn theo khóa

In [4]:
cnt = defaultdict(int)
for who, w in [("Alice", 2), ("Bob", 3), ("Alice", 3)]:
    cnt[who] += w
print(dict(cnt))
c2 = Counter(); c2["x"] += 5; print(c2, c2["chưa có"])      # Counter trả 0 cho khóa chưa có

{'Alice': 5, 'Bob': 3}
Counter({'x': 5}) 0


### 3.4 So sánh chuỗi trong Python: theo mã Unicode từng ký tự

So sánh từ trái sang phải, ký tự đầu tiên khác nhau quyết định; nếu một chuỗi là tiền tố của chuỗi kia thì chuỗi **ngắn hơn** nhỏ hơn.

In [5]:
print(ord('A'), ord('Z'), ord('a'), ord('z'))
pairs = [("Alice", "alice"), ("Zed", "abe"), ("Bob", "Bobby"), ("userThree", "userTwo"), ("Charlie", "Bob")]
for x, y in pairs:
    print(f"{x!r} < {y!r}: {x < y}")

65 90 97 122
'Alice' < 'alice': True
'Zed' < 'abe': True
'Bob' < 'Bobby': True
'userThree' < 'userTwo': True
'Charlie' < 'Bob': False


`"Zed" < "abe"` là **True** vì `'Z' = 90 < 'a' = 97` — đúng ghi chú "chữ hoa đứng trước chữ thường". Vì vậy chỉ cần dùng phép so sánh chuỗi mặc định, **không** được `lower()`.

### 3.5 So sánh tuple: so phần tử đầu, hòa thì so phần tử sau

In [6]:
print((5, "Bob") < (5, "Charlie"), (6, "Bob") > (5, "Zoe"), max([(5, "Bob"), (5, "Charlie"), (3, "Zoe")]))

True True (5, 'Charlie')


## 4. Ý tưởng nền tảng / trực giác

Bài gồm hai bước tách biệt:

1. **Tổng hợp:** với mỗi tin nhắn `i`, cộng `words(messages[i])` vào `total[senders[i]]`. Sau một lượt, `total` chứa số từ của từng người.
2. **Chọn:** người có `(total, name)` **lớn nhất** theo thứ tự tuple. Tiêu chí chính so `total`; khi hòa, tuple tự so `name` — đúng quy tắc hòa của đề.

Tính tay ví dụ 1:

| i | sender | message | số từ | total sau bước |
|---|---|---|---|---|
| 0 | Alice | Hello userTwooo | 2 | Alice 2 |
| 1 | userTwo | Hi userThree | 2 | Alice 2, userTwo 2 |
| 2 | userThree | Wonderful day Alice | 3 | …, userThree 3 |
| 3 | Alice | Nice day userThree | 3 | **Alice 5**, userTwo 2, userThree 3 |

Khóa: `(5,"Alice") > (3,"userThree") > (2,"userTwo")` ⇒ `"Alice"`. Chú ý: tin nhắn số 0 nhắc chữ `"userTwooo"` và tin số 2 nhắc `"Alice"` — **nội dung** tin nhắn không liên quan đến ai gửi; chỉ `senders[i]` quyết định.

Ví dụ 2: Bob 5, Charlie 5 ⇒ so `(5,"Bob")` với `(5,"Charlie")` ⇒ `"Charlie"` lớn hơn (`'C' > 'B'`).

In [7]:
def trace_aggregate(messages, senders):
    total = {}
    for i, (m_, s_) in enumerate(zip(messages, senders)):
        w = m_.count(" ") + 1
        total[s_] = total.get(s_, 0) + w
        print(f"i={i} {s_:<10} +{w} -> {total}")
    best = max(total, key=lambda k: (total[k], k))
    print("khóa:", sorted(((v, k) for k, v in total.items()), reverse=True), "=> chọn", best)
    return best

trace_aggregate(*examples[0][:2])

i=0 Alice      +2 -> {'Alice': 2}
i=1 userTwo    +2 -> {'Alice': 2, 'userTwo': 2}
i=2 userThree  +3 -> {'Alice': 2, 'userTwo': 2, 'userThree': 3}
i=3 Alice      +3 -> {'Alice': 5, 'userTwo': 2, 'userThree': 3}
khóa: [(5, 'Alice'), (3, 'userThree'), (2, 'userTwo')] => chọn Alice


'Alice'

## 5. Cách 1 — Brute Force / baseline

### 5.1 Với mỗi người gửi phân biệt, quét lại toàn bộ nhật ký — O(u · n · L)

`u` = số người gửi phân biệt, `L` = độ dài tin nhắn. Đây là cách "làm theo định nghĩa", không cần cấu trúc dữ liệu.

In [8]:
def brute(messages, senders):
    best_name, best_cnt = None, -1
    for name in set(senders):
        c = sum(len(m_.split()) for m_, s_ in zip(messages, senders) if s_ == name)
        if c > best_cnt or (c == best_cnt and name > best_name):
            best_name, best_cnt = name, c
    return best_name

for msgs, snd, exp in examples:
    got = brute(msgs, snd)
    print(got, "OK" if got == exp else "SAI")

Alice OK
Charlie OK


**Độ phức tạp:** worst `u = n = 10⁴` ⇒ `10⁸` phép so sánh tên — quá chậm trong Python. Lặp `set(senders)` không có thứ tự cố định, nhưng điều kiện chọn đã xử lý hòa tường minh nên kết quả vẫn xác định.

### 5.2 Baseline 2 — Sắp xếp theo người gửi rồi gom các đoạn liên tiếp — O(n log n)

In [9]:
def sort_group(messages, senders):
    pairs = sorted(zip(senders, messages))
    best = (-1, "")
    i = 0
    while i < len(pairs):
        j, c = i, 0
        while j < len(pairs) and pairs[j][0] == pairs[i][0]:
            c += pairs[j][1].count(" ") + 1; j += 1
        best = max(best, (c, pairs[i][0]))
        i = j
    return best[1]

for msgs, snd, exp in examples:
    print(sort_group(msgs, snd), "OK" if sort_group(msgs, snd) == exp else "SAI")

Alice OK
Charlie OK


Cách này không cần hash map nhưng tốn O(n log n) do sắp xếp, và việc so sánh tuple `(tên, tin nhắn)` khi sắp xếp còn so cả nội dung tin nhắn nếu trùng tên (tốn thêm).

**Nút thắt:** ta chỉ cần **một lượt** cộng dồn theo khóa — đó là việc của bảng băm, O(1) trung bình mỗi lần cập nhật.

## 6. Ý tưởng thuật toán / bước đột phá

**Bảng băm + chọn theo khóa ghép.**

- Lượt 1 (O(tổng độ dài)): `cnt[sender] += message.count(" ") + 1`.
- Lượt 2 (O(u · |tên|)): giữ `ans`; **nếu** `cnt[k] > cnt[ans]` **hoặc** (`cnt[k] == cnt[ans]` **và** `k > ans`) **thì** `ans = k`. Tương đương `max(cnt, key=lambda k: (cnt[k], k))`.

**Có thể gộp hai lượt thành một không?** Có, nhưng phải cẩn thận: khi cập nhật `cnt[s]`, chỉ người `s` thay đổi (và chỉ **tăng**), nên chỉ cần so `(cnt[s], s)` với `(cnt[ans], ans)` sau mỗi cập nhật. Vì tổng của các người khác không đổi và tổng chỉ tăng, người tốt nhất sau mỗi bước luôn là `ans` cũ hoặc `s` ⇒ đúng (xem cách 3).

**Vì sao không cần xử lý chuỗi tinh vi:** số từ = số dấu cách + 1, đề đảm bảo định dạng chuẩn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` + vòng chọn tường minh (cách doocs)

In [10]:
def counter_scan(messages, senders):
    cnt = Counter()
    for message, sender in zip(messages, senders):
        cnt[sender] += message.count(" ") + 1
    ans = senders[0]
    for k, v in cnt.items():
        if cnt[ans] < v or (cnt[ans] == v and ans < k):
            ans = k
    return ans

for msgs, snd, exp in examples:
    print(counter_scan(msgs, snd), "OK" if counter_scan(msgs, snd) == exp else "SAI")

Alice OK
Charlie OK


### 7.2 Cách 3 — Một lượt duy nhất (cập nhật người tốt nhất ngay khi cộng)

In [11]:
def one_pass(messages, senders):
    cnt = {}; ans = None
    for message, sender in zip(messages, senders):
        cnt[sender] = cnt.get(sender, 0) + message.count(" ") + 1
        if ans is None or (cnt[sender], sender) > (cnt[ans], ans):
            ans = sender
    return ans

for msgs, snd, exp in examples:
    print(one_pass(msgs, snd), "OK" if one_pass(msgs, snd) == exp else "SAI")

Alice OK
Charlie OK


Lập luận đúng của cách 3: gọi `best_t` là người có khóa lớn nhất sau bước `t`. Ở bước `t+1`, chỉ khóa của `s = senders[t]` thay đổi và **tăng**. Mọi người khác `x ≠ s` có khóa không đổi ≤ khóa của `best_t`. Vậy người tốt nhất mới ∈ {`best_t`, `s`} — nếu `best_t == s` thì khóa của nó tăng, vẫn là tốt nhất. ⇒ chỉ cần so `s` với `ans`.

Lưu ý: tính chất này **phụ thuộc** vào việc tổng chỉ tăng (số từ ≥ 1). Nếu có thể trừ (ví dụ xóa tin nhắn), một lượt không còn đúng — xem biến thể 15.3.

### 7.3 Cách 4 — `max` với khóa tuple (gọn nhất)

In [12]:
def max_key(messages, senders):
    cnt = defaultdict(int)
    for m_, s_ in zip(messages, senders):
        cnt[s_] += len(m_.split())
    return max(cnt, key=lambda k: (cnt[k], k))

for msgs, snd, exp in examples:
    print(max_key(msgs, snd), "OK" if max_key(msgs, snd) == exp else "SAI")

Alice OK
Charlie OK


### 7.4 Cách 5 — Sắp xếp các cặp `(tổng, tên)` giảm dần

Hữu ích khi cần **bảng xếp hạng** đầy đủ chứ không chỉ người đứng đầu.

In [13]:
def ranking(messages, senders):
    cnt = Counter()
    for m_, s_ in zip(messages, senders):
        cnt[s_] += m_.count(" ") + 1
    return sorted(cnt.items(), key=lambda kv: (kv[1], kv[0]), reverse=True)

r = ranking(*examples[0][:2]); print(r); print("đứng đầu:", r[0][0])

[('Alice', 5), ('userThree', 3), ('userTwo', 2)]
đứng đầu: Alice


## 8. TRACE từng bước

### 8.1 Trace cách 3 (một lượt) trên một nhật ký có hòa và chữ hoa/thường

In [14]:
msgs = ["a b c", "x y", "p q r", "z", "k l m n", "u v"]
snds = ["bob", "Bob", "Bob", "bob", "alice", "Zara"]

def trace_one_pass(messages, senders):
    cnt = {}; ans = None
    print(f"{'i':>2} {'sender':<7} {'+từ':>3} {'cnt[sender]':>11}  so sánh khóa                    ans")
    for i, (m_, s_) in enumerate(zip(messages, senders)):
        w = m_.count(" ") + 1
        cnt[s_] = cnt.get(s_, 0) + w
        cmp = "—" if ans is None else f"{(cnt[s_], s_)} vs {(cnt[ans], ans)}"
        if ans is None or (cnt[s_], s_) > (cnt[ans], ans):
            ans = s_
        print(f"{i:>2} {s_:<7} {w:>3} {cnt[s_]:>11}  {cmp:<30}  {ans}")
    return ans

print("kết quả:", trace_one_pass(msgs, snds), "| brute:", brute(msgs, snds))

 i sender  +từ cnt[sender]  so sánh khóa                    ans
 0 bob       3           3  —                               bob
 1 Bob       2           2  (2, 'Bob') vs (3, 'bob')        bob
 2 Bob       3           5  (5, 'Bob') vs (3, 'bob')        Bob
 3 bob       1           4  (4, 'bob') vs (5, 'Bob')        Bob
 4 alice     4           4  (4, 'alice') vs (5, 'Bob')      Bob
 5 Zara      2           2  (2, 'Zara') vs (5, 'Bob')       Bob
kết quả: Bob | brute: Bob


Diễn giải: sau bước 2, `"Bob"` có 5 và `"bob"` có 3. Bước 3 `"bob"` lên 4 — vẫn thua. Bước 4 `"alice"` lên 4 — thua. Bước 5 `"Zara"` có 2. Kết quả `"Bob"`. Nếu `"bob"` cũng có 5, ta sẽ chọn `"bob"` vì `'b' (98) > 'B' (66)`.

### 8.2 Trace trường hợp hòa thuần túy

In [15]:
msgs2 = ["one two", "three four", "five six"]
snds2 = ["Zoe", "adam", "Mia"]
print("kết quả:", trace_one_pass(msgs2, snds2))
print("sắp xếp tên tăng dần:", sorted(snds2), "-> lớn nhất là", max(snds2))

 i sender  +từ cnt[sender]  so sánh khóa                    ans
 0 Zoe       2           2  —                               Zoe
 1 adam      2           2  (2, 'adam') vs (2, 'Zoe')       adam
 2 Mia       2           2  (2, 'Mia') vs (2, 'adam')       adam
kết quả: adam
sắp xếp tên tăng dần: ['Mia', 'Zoe', 'adam'] -> lớn nhất là adam


Cả ba người hòa 2 từ. Thứ tự từ điển: `"Mia" < "Zoe" < "adam"` (chữ thường sau chữ hoa) ⇒ đáp án `"adam"` — dù theo cảm giác "bảng chữ cái" thì `adam` đứng đầu. Đây là ví dụ tự dựng hay bị làm sai nhất.

## 9. Code nộp LeetCode

In [16]:
from typing import List
from collections import Counter

class Solution:
    def largestWordCount(self, messages: List[str], senders: List[str]) -> str:
        cnt = Counter()
        for message, sender in zip(messages, senders):
            cnt[sender] += message.count(" ") + 1     # số từ = số dấu cách + 1
        # số từ lớn nhất; hòa thì tên lớn nhất (so sánh chuỗi mặc định: hoa < thường)
        return max(cnt, key=lambda k: (cnt[k], k))

In [17]:
sol = Solution()
for msgs, snd, exp in examples:
    got = sol.largestWordCount(msgs, snd)
    print(repr(got), "| kỳ vọng", repr(exp), "| OK" if got == exp else "| SAI")

'Alice' | kỳ vọng 'Alice' | OK
'Charlie' | kỳ vọng 'Charlie' | OK


## 10. Vì sao thuật toán đúng

**Bước tổng hợp.** Gọi `W(x) = Σ words(messages[i])` trên các `i` có `senders[i] = x`. Mỗi chỉ số `i` được xử lý đúng một lần và cộng `words(messages[i])` vào đúng khóa `senders[i]` ⇒ sau vòng lặp `cnt[x] = W(x)` với mọi người gửi `x` (bất biến: sau khi xử lý `i` tin nhắn đầu, `cnt[x]` = tổng số từ của `x` trong `i` tin đó).

**Đếm từ.** Với câu chuẩn gồm các từ `w₁ … w_k` (mỗi từ khác rỗng, vì chỉ có một dấu cách giữa hai từ và không có dấu cách đầu/cuối), câu là `w₁ + " " + w₂ + … + " " + w_k`, có đúng `k − 1` dấu cách ⇒ `count(" ") + 1 = k`. Tin nhắn có độ dài ≥ 1 nên `k ≥ 1`.

**Bước chọn.** Đề định nghĩa người thắng là người có `W` lớn nhất, hòa thì tên lớn nhất. Đó chính là phần tử lớn nhất theo **thứ tự từ điển trên cặp** `(W(x), x)`: `(a, x) > (b, y)` ⇔ `a > b` hoặc (`a = b` và `x > y`). Thứ tự này là **toàn phần** trên các người gửi (tên khác nhau ⇒ không có hai cặp bằng nhau) ⇒ `max` trả về **duy nhất** một người, không phụ thuộc thứ tự duyệt dict.

**Thứ tự chuỗi.** Python so sánh chuỗi theo điểm mã Unicode; với chữ cái ASCII, mọi chữ hoa (65–90) nhỏ hơn mọi chữ thường (97–122) — đúng ghi chú của đề. Quy tắc "tiền tố ngắn hơn thì nhỏ hơn" cũng là quy tắc từ điển chuẩn. ∎

## 11. Phân tích Time / Space Complexity

Gọi `S = Σ len(messages[i]) ≤ 10⁶`, `u` = số người gửi phân biệt (≤ n), tên dài ≤ 10.

| Cách | Time | Space |
|---|---|---|
| 1a. Quét lại cho mỗi người | O(u · S) | O(u) |
| 1b. Sắp xếp + gom | O(n log n · 10 + S) | O(n) |
| 2. Counter + vòng chọn | O(S + u) | O(u) |
| 3. Một lượt | O(S) | O(u) |
| 4. `max` khóa tuple | O(S + u) | O(u) |
| 5. Bảng xếp hạng | O(S + u log u) | O(u) |

`message.count(" ")` chạy trong C, nhanh hơn `split()` (không tạo danh sách từ).

In [18]:
rng = random.Random(2284)
def rand_name(k=10):
    return "".join(rng.choice(string.ascii_letters) for _ in range(rng.randint(1, k)))
def rand_msg():
    words = []; length = -1
    while True:
        w = "".join(rng.choice(string.ascii_letters) for _ in range(rng.randint(1, 6)))
        if length + 1 + len(w) > 100: break
        words.append(w); length += 1 + len(w)
        if rng.random() < 0.15: break
    return " ".join(words)

names = [rand_name() for _ in range(2000)]
big_m = [rand_msg() for _ in range(10**4)]; big_s = [rng.choice(names) for _ in range(10**4)]
res = []
for name, f in [("counter_scan", counter_scan), ("one_pass", one_pass), ("max_key", max_key),
                ("sort_group", sort_group), ("Solution", Solution().largestWordCount)]:
    t0 = time.perf_counter(); res.append(f(big_m, big_s)); dt = time.perf_counter() - t0
    print(f"{name:<13} {dt*1000:7.2f} ms -> {res[-1]}")
assert len(set(res)) == 1
t0 = time.perf_counter(); b = brute(big_m[:2000], big_s[:2000]); print(f"brute (n=2000): {(time.perf_counter()-t0)*1000:.0f} ms")

counter_scan     1.90 ms -> a
one_pass         1.75 ms -> a
max_key          3.61 ms -> a
sort_group       6.19 ms -> a
Solution         2.54 ms -> a


brute (n=2000): 58 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute": brute, "sort_group": sort_group, "counter_scan": counter_scan, "one_pass": one_pass,
       "max_key": max_key, "ranking": lambda m_, s_: ranking(m_, s_)[0][0], "Solution": Solution().largestWordCount}
edge = [(m_, s_, e) for m_, s_, e in examples] + [
    (["hi"], ["A"], "A"),                                         # n = 1
    (["a b", "c d"], ["Zoe", "adam"], "adam"),                    # hòa, chữ thường lớn hơn chữ hoa
    (["a b", "c d"], ["alice", "Alice"], "alice"),                # khác hoa/thường là 2 người
    (["a b c", "d"], ["Bob", "Bobby"], "Bob"),                    # nhiều từ hơn thắng dù tên ngắn
    (["a b", "c d"], ["Bob", "Bobby"], "Bobby"),                  # hòa: tiền tố nhỏ hơn
    (["x"] * 5 + ["a b c d e"], ["m"] * 5 + ["n"], "n"),          # nhiều tin ngắn vs một tin dài: hòa 5-5
    (["a b c d e f g h i j"] * 3, ["q", "q", "r"], "q"),          # cộng dồn nhiều tin
    (["Alice Bob Carol"], ["dave"], "dave"),                      # nội dung chứa tên người khác — không liên quan
]
for msgs, snd, exp in edge:
    for name, f in ALL.items():
        got = f(msgs, snd)
        assert got == exp, (name, msgs, snd, got, exp)
    print(f"{str(snd):<36} -> {exp!r}")
print("edge OK")

['Alice', 'userTwo', 'userThree', 'Alice'] -> 'Alice'
['Bob', 'Charlie']                   -> 'Charlie'
['A']                                -> 'A'
['Zoe', 'adam']                      -> 'adam'
['alice', 'Alice']                   -> 'alice'
['Bob', 'Bobby']                     -> 'Bob'
['Bob', 'Bobby']                     -> 'Bobby'
['m', 'm', 'm', 'm', 'm', 'n']       -> 'n'
['q', 'q', 'r']                      -> 'q'
['dave']                             -> 'dave'
edge OK


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

Bảng tên nhỏ để có nhiều lần một người gửi nhiều tin, và nhiều trường hợp hòa.

In [20]:
rng = random.Random(42)
pool = ["a", "A", "b", "B", "ab", "Ab", "aB", "ba", "z", "Z"]
for _ in range(3000):
    n = rng.randint(1, 8)
    snd = [rng.choice(pool) for _ in range(n)]
    msgs = [" ".join("w" * rng.randint(1, 3) for _ in range(rng.randint(1, 4))) for _ in range(n)]
    ref = brute(msgs, snd)
    for name, f in ALL.items():
        assert f(msgs, snd) == ref, (name, msgs, snd)
print("3000 nhật ký ngẫu nhiên OK")

3000 nhật ký ngẫu nhiên OK


### 12.3 Tính chất: hoán vị thứ tự tin nhắn không đổi đáp án; thêm tin của người thắng không đổi đáp án

In [21]:
rng = random.Random(99)
for _ in range(1000):
    n = rng.randint(1, 8)
    snd = [rng.choice(pool) for _ in range(n)]
    msgs = [" ".join("w" for _ in range(rng.randint(1, 4))) for _ in range(n)]
    ans = Solution().largestWordCount(msgs, snd)
    idx = list(range(n)); rng.shuffle(idx)
    assert Solution().largestWordCount([msgs[i] for i in idx], [snd[i] for i in idx]) == ans
    assert Solution().largestWordCount(msgs + ["x y"], snd + [ans]) == ans
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: `n = 10⁴` tin, mỗi tin 50 từ, mọi người hòa

In [22]:
msgs = [" ".join(["a"] * 50)] * 10**4
snds = [f"u{i % 100:02d}" for i in range(10**4)]               # 100 người, mỗi người 100 tin * 50 từ
print(Solution().largestWordCount(msgs, snds), "(kỳ vọng u99: mọi người hòa 5000 từ, tên lớn nhất)")
assert Solution().largestWordCount(msgs, snds) == "u99"

u99 (kỳ vọng u99: mọi người hòa 5000 từ, tên lớn nhất)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `max(cnt, key=cnt.get)`: khi hòa trả về người **xuất hiện trước**

In [23]:
def wrong_max_get(messages, senders):
    cnt = Counter()
    for m_, s_ in zip(messages, senders): cnt[s_] += m_.count(" ") + 1
    return max(cnt, key=cnt.get)                 # SAI: không xử lý hòa theo tên

print("bản sai :", wrong_max_get(*examples[1][:2]), "| bản đúng:", Solution().largestWordCount(*examples[1][:2]))

bản sai : Bob | bản đúng: Charlie


**Vì sao sai:** `max` trả về phần tử lớn nhất **đầu tiên** gặp; dict giữ thứ tự chèn nên `"Bob"` (chèn trước) thắng. **Sửa:** khóa tuple `(cnt[k], k)`.

### Lỗi 2 — So sánh tên không phân biệt hoa/thường (`lower()`) hoặc theo "bảng chữ cái thường"

In [24]:
def wrong_lower(messages, senders):
    cnt = Counter()
    for m_, s_ in zip(messages, senders): cnt[s_] += m_.count(" ") + 1
    return max(cnt, key=lambda k: (cnt[k], k.lower()))     # SAI

cases = [(["a b", "c d"], ["Zoe", "adam"]), (["a b", "c d"], ["Alice", "alice"])]
for m_, s_ in cases:
    print("bản sai :", wrong_lower(m_, s_), "| bản đúng:", Solution().largestWordCount(m_, s_))

bản sai : Zoe | bản đúng: adam
bản sai : Alice | bản đúng: alice


**Vì sao sai:** đề nói rõ chữ hoa đứng trước chữ thường và `"Alice" ≠ "alice"`. Với `"Alice"`/`"alice"` (Alice gửi trước), `lower()` làm hai khóa bằng nhau và `max` lấy người chèn trước là `"Alice"` — kết quả phụ thuộc thứ tự tin nhắn, trong khi đáp án đúng luôn là `"alice"` (`'a' > 'A'`). **Sửa:** so sánh chuỗi gốc.

### Lỗi 3 — Ghi đè thay vì cộng dồn (`cnt[s] = words` thay vì `+=`)

In [25]:
def wrong_overwrite(messages, senders):
    cnt = {}
    for m_, s_ in zip(messages, senders): cnt[s_] = m_.count(" ") + 1     # SAI
    return max(cnt, key=lambda k: (cnt[k], k))

print("bản sai :", wrong_overwrite(*examples[0][:2]), "| bản đúng:", Solution().largestWordCount(*examples[0][:2]))

bản sai : userThree | bản đúng: Alice


**Vì sao sai:** Alice chỉ còn 3 (tin cuối) thay vì 5 ⇒ hòa với userThree, `"userThree" > "Alice"` ⇒ sai. **Sửa:** `+=`.

### Lỗi 4 — Đếm ký tự (`len(message)`) thay vì đếm từ

In [26]:
def wrong_len(messages, senders):
    cnt = Counter()
    for m_, s_ in zip(messages, senders): cnt[s_] += len(m_)             # SAI
    return max(cnt, key=lambda k: (cnt[k], k))

m_, s_ = ["Supercalifragilistic", "a b c"], ["Mary", "Bert"]
print("bản sai :", wrong_len(m_, s_), "| bản đúng:", Solution().largestWordCount(m_, s_))

bản sai : Mary | bản đúng: Bert


### Lỗi 5 — Chọn theo **tin nhắn dài nhất** (không gom theo người)

In [27]:
def wrong_per_message(messages, senders):
    best = max(range(len(messages)), key=lambda i: (messages[i].count(" ") + 1, senders[i]))
    return senders[best]                                                  # SAI

print("bản sai :", wrong_per_message(*examples[0][:2]), "| bản đúng:", Solution().largestWordCount(*examples[0][:2]))

bản sai : userThree | bản đúng: Alice


**Vì sao sai:** tin dài nhất có 3 từ (của Alice và userThree) ⇒ chọn `"userThree"` (tên lớn hơn), trong khi tổng của Alice là 5. **Sửa:** cộng dồn theo người gửi trước khi chọn.

### Lỗi 6 — Chọn tên **nhỏ nhất** khi hòa (đảo dấu)

In [28]:
def wrong_min_name(messages, senders):
    cnt = Counter()
    for m_, s_ in zip(messages, senders): cnt[s_] += m_.count(" ") + 1
    return min(cnt, key=lambda k: (-cnt[k], k))                          # SAI: hòa lấy tên nhỏ nhất

print("bản sai :", wrong_min_name(*examples[1][:2]), "| bản đúng:", Solution().largestWordCount(*examples[1][:2]))

bản sai : Bob | bản đúng: Charlie


**Sửa:** muốn "lớn nhất ở cả hai tiêu chí" thì `max(key=(cnt, name))`; muốn dùng `min` thì phải đảo cả hai — với chuỗi không đảo dấu được, nên dùng `max`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng theo nhóm rồi chọn lớn nhất, hòa thì theo khóa phụ" ⇒ **hash map + `max` với khóa tuple**.
- Câu chuẩn ⇒ `count(" ") + 1`; dữ liệu có thể bẩn ⇒ `len(s.split())`.
- Hòa theo tên **lớn nhất**: `max(key=(score, name))`; hòa theo tên **nhỏ nhất**: `min(key=(-score, name))` hoặc `sorted(key=(-score, name))[0]`.
- Không dùng `max(d, key=d.get)` khi đề có quy tắc hòa.
- Python so sánh chuỗi theo mã Unicode ⇒ hoa < thường; đừng `lower()` khi đề phân biệt hoa/thường.
- Bài cùng họ: 692 Top K Frequent Words (hòa theo thứ tự từ điển **nhỏ**), 819 Most Common Word, 1636 Sort Array by Increasing Frequency, 2273 Find Resultant Array After Removing Anagrams, 1817 Finding the Users Active Minutes.

## 15. Biến thể của bài

### 15.1 Biến thể A — Top-k người gửi (hòa: tên lớn hơn đứng trước) bằng heap

In [29]:
def top_k_senders(messages, senders, k):
    cnt = Counter()
    for m_, s_ in zip(messages, senders): cnt[s_] += m_.count(" ") + 1
    return [name for _, name in heapq.nlargest(k, ((v, n) for n, v in cnt.items()))]

rng = random.Random(151)
for _ in range(300):
    n = rng.randint(1, 10); snd = [rng.choice(pool) for _ in range(n)]
    msgs = [" ".join("w" for _ in range(rng.randint(1, 3))) for _ in range(n)]
    k = rng.randint(1, len(set(snd)))
    assert top_k_senders(msgs, snd, k) == [x for x, _ in ranking(msgs, snd)][:k]
print(top_k_senders(*examples[0][:2], 2))

['Alice', 'userThree']


`heapq.nlargest(k, ...)` chạy O(u log k) — tốt hơn sắp xếp toàn bộ khi `k ≪ u`.

### 15.2 Biến thể B — Hòa thì lấy tên **nhỏ nhất** (kiểu LC 692)

In [30]:
def largest_tie_smallest(messages, senders):
    cnt = Counter()
    for m_, s_ in zip(messages, senders): cnt[s_] += m_.count(" ") + 1
    return min(cnt, key=lambda k: (-cnt[k], k))

print(largest_tie_smallest(*examples[1][:2]), "(Bob và Charlie hòa -> Bob)")

Bob (Bob và Charlie hòa -> Bob)


### 15.3 Biến thể C — Dòng sự kiện có **xóa tin nhắn**, hỏi người dẫn đầu sau mỗi sự kiện

Khi tổng có thể giảm, cách một lượt (7.2) không còn đúng: người dẫn đầu bị trừ có thể rơi xuống. Dùng `SortedList` các cặp `(tổng, tên)` — cập nhật O(log u), truy vấn O(1).

In [31]:
from sortedcontainers import SortedList

def leader_stream(events):
    # events: ("add", sender, message) | ("del", sender, message)
    cnt = defaultdict(int); sl = SortedList(); out = []
    for op, s_, m_ in events:
        if cnt[s_] > 0: sl.remove((cnt[s_], s_))
        cnt[s_] += (1 if op == "add" else -1) * (m_.count(" ") + 1)
        if cnt[s_] > 0: sl.add((cnt[s_], s_))
        out.append(sl[-1][1] if sl else None)
    return out

def leader_stream_brute(events):
    cnt = defaultdict(int); out = []
    for op, s_, m_ in events:
        cnt[s_] += (1 if op == "add" else -1) * (m_.count(" ") + 1)
        alive = [k for k, v in cnt.items() if v > 0]
        out.append(max(alive, key=lambda k: (cnt[k], k)) if alive else None)
    return out

rng = random.Random(153)
for _ in range(300):
    live = []; ev = []
    for _ in range(rng.randint(1, 15)):
        if live and rng.random() < 0.35:
            s_, m_ = live.pop(rng.randrange(len(live))); ev.append(("del", s_, m_))
        else:
            s_ = rng.choice(pool[:5]); m_ = " ".join("w" for _ in range(rng.randint(1, 3)))
            live.append((s_, m_)); ev.append(("add", s_, m_))
    assert leader_stream(ev) == leader_stream_brute(ev)
demo = [("add", "Bob", "a b c"), ("add", "Al", "a b"), ("add", "Al", "c d"), ("del", "Al", "a b"), ("del", "Bob", "a b c")]
print(leader_stream(demo))

['Bob', 'Bob', 'Al', 'Bob', 'Al']


### 15.4 Biến thể D — Tin nhắn **không chuẩn** (nhiều dấu cách, dấu câu)

Khi dữ liệu bẩn, `count(" ") + 1` đếm sai; dùng `split()` (bỏ khoảng trắng thừa) hoặc biểu thức chính quy để chỉ đếm cụm chữ cái.

In [32]:
import re
def words_dirty(s):
    return len(re.findall(r"[A-Za-z]+", s))
for s in ["  Hello   world  ", "Hi, there!", "...", "one"]:
    print(f"{s!r:<20} count+1={s.count(' ') + 1:<3} split()={len(s.split()):<3} regex={words_dirty(s)}")

'  Hello   world  '  count+1=8   split()=2   regex=2
'Hi, there!'         count+1=2   split()=2   regex=2
'...'                count+1=1   split()=1   regex=0
'one'                count+1=1   split()=1   regex=1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Quét lại mỗi người | định nghĩa | O(u·S) | O(u) | kiểm chứng |
| 1b. Sắp xếp + gom | không cần hash | O(n log n + S) | O(n) | khi cần nhóm có thứ tự |
| 2. Counter + vòng chọn | doocs | O(S + u) | O(u) | tường minh |
| 3. Một lượt | tổng chỉ tăng | O(S) | O(u) | dữ liệu dạng dòng (stream) chỉ thêm |
| 4. `max` khóa tuple | gọn | O(S + u) | O(u) | **mặc định** |
| 5. Bảng xếp hạng / heap | sắp xếp | O(S + u log u) | O(u) | cần top-k |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `largestWordCount(["a b", "c d"], ["Zoe", "zoe"])` và `largestWordCount(["a", "b c"], ["x", "x"])`?

**Bài 2 (tính tay):** `messages = ["k", "k k", "k k k", "k k"]`, `senders = ["M", "m", "M", "Mm"]`. Tính tổng từng người và đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(messages, senders)` trả về **danh sách mọi người** có số từ lớn nhất, sắp xếp tăng dần theo tên.

**Bài 4 (tự viết lại):** Viết `bt4_todo(messages, senders)` cài bản nộp không nhìn code, **không** dùng `Counter`/`defaultdict`, **không** dùng `max` (chỉ vòng lặp và `dict`).

**Bài 5 (mini challenge):** Viết `bt5_todo(messages, senders)` trả về danh sách `leader[i]` = người dẫn đầu sau khi xử lý tin nhắn thứ `i` (áp dụng quy tắc hòa của đề), chạy O(S).

In [33]:
def bt3_todo(messages, senders):
    # TODO: danh sách mọi người có số từ lớn nhất, sắp xếp tăng dần
    pass

def bt4_todo(messages, senders):
    # TODO: bản nộp chỉ dùng dict và vòng lặp
    pass

def bt5_todo(messages, senders):
    # TODO: người dẫn đầu sau mỗi tin nhắn
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: hai tên khác nhau vì hoa/thường. Mức 2: hòa 2–2, so `"Zoe"` với `"zoe"`. Mức 3: `'z' > 'Z'` ⇒ `"zoe"`; câu hai: `x` có 1 + 2 = 3 ⇒ `"x"`.

**Bài 2.** Mức 1: M = 1 + 3 = 4. Mức 2: m = 2, Mm = 2. Mức 3: `"M"` (4 lớn nhất, không hòa).

**Bài 3.** Mức 1: tính `cnt`. Mức 2: `best = max(cnt.values())`. Mức 3: `sorted(k for k, v in cnt.items() if v == best)`.

**Bài 4.** Mức 1: `cnt[s] = cnt.get(s, 0) + words`. Mức 2: duyệt `cnt.items()` giữ `ans`. Mức 3: điều kiện `v > cnt[ans] or (v == cnt[ans] and k > ans)`.

**Bài 5.** Mức 1: tổng chỉ tăng. Mức 2: người dẫn đầu mới ∈ {người cũ, người vừa gửi}. Mức 3: so khóa `(cnt[s], s)` với `(cnt[ans], ans)` rồi `append(ans)`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `"zoe"`, `"x"`. **Bài 2:** `"M"`.

In [34]:
assert Solution().largestWordCount(["a b", "c d"], ["Zoe", "zoe"]) == "zoe"
assert Solution().largestWordCount(["a", "b c"], ["x", "x"]) == "x"
assert Solution().largestWordCount(["k", "k k", "k k k", "k k"], ["M", "m", "M", "Mm"]) == "M"

def bt3(messages, senders):
    cnt = Counter()
    for m_, s_ in zip(messages, senders): cnt[s_] += m_.count(" ") + 1
    best = max(cnt.values())
    return sorted(k for k, v in cnt.items() if v == best)

def bt4(messages, senders):
    cnt = {}
    for m_, s_ in zip(messages, senders):
        cnt[s_] = cnt.get(s_, 0) + m_.count(" ") + 1
    ans = senders[0]
    for k, v in cnt.items():
        if v > cnt[ans] or (v == cnt[ans] and k > ans):
            ans = k
    return ans

def bt5(messages, senders):
    cnt = {}; ans = None; out = []
    for m_, s_ in zip(messages, senders):
        cnt[s_] = cnt.get(s_, 0) + m_.count(" ") + 1
        if ans is None or (cnt[s_], s_) > (cnt[ans], ans): ans = s_
        out.append(ans)
    return out

rng = random.Random(1984)
for _ in range(1000):
    n = rng.randint(1, 8); snd = [rng.choice(pool) for _ in range(n)]
    msgs = [" ".join("w" for _ in range(rng.randint(1, 4))) for _ in range(n)]
    assert bt3(msgs, snd)[-1] == brute(msgs, snd)
    assert bt4(msgs, snd) == brute(msgs, snd)
    assert bt5(msgs, snd) == [brute(msgs[:i + 1], snd[:i + 1]) for i in range(n)]
print(bt3(*examples[1][:2]), bt5(*examples[0][:2]))
print("Đáp án bài tập: tất cả assert OK")

['Bob', 'Charlie'] ['Alice', 'userTwo', 'userThree', 'Alice']
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cnt = Counter()
for msg, who in zip(messages, senders):
    cnt[who] += msg.count(" ") + 1          # số từ = số dấu cách + 1
return max(cnt, key=lambda k: (cnt[k], k))  # nhiều từ nhất, hòa -> tên lớn nhất
```

**Checklist:**
- [ ] Cộng dồn (`+=`) theo **người gửi**, không theo tin nhắn.
- [ ] Đếm **từ**, không đếm ký tự.
- [ ] Hòa ⇒ tên **lớn nhất**; khóa tuple `(cnt, name)` với `max`.
- [ ] Không `lower()`; chữ hoa < chữ thường đúng như so sánh chuỗi mặc định.
- [ ] Không dùng `max(cnt, key=cnt.get)`.